# LeetCode 429 — N-ary Tree Level Order Traversal bằng Python

Link đề: https://leetcode.com/problems/n-ary-tree-level-order-traversal/ · Độ khó: **Medium** · Chủ đề: Tree, Breadth-First Search (BFS), Queue

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children
"""

class Solution:
    def levelOrder(self, root: 'Node') -> List[List[int]]:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách trên hàng trăm cây ngẫu nhiên với một "oracle" độc lập tính từ mảng cha). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho **cây N-phân** (mỗi nút có **một danh sách con** với số con bất kỳ); trả về các giá trị **theo từng tầng**, mỗi tầng một list, từ **trên xuống dưới**, trong tầng thì **từ trái sang phải**.
2. Nắm các công cụ Python: lớp `Node` có `children` là **list**, **hàng đợi** `collections.deque` (`append`, `popleft`, `extend`), vì sao `list.pop(0)` chậm, **list comprehension** nhiều tầng, và cách **dựng cây từ list kiểu LeetCode** (các nhóm con ngăn cách bởi `null`).
3. Đi từ **thu độ sâu của mọi nút rồi gom nhóm** → **BFS bằng hàng đợi, "chụp" kích thước tầng** → **BFS theo "frontier" (danh sách tầng hiện tại)** → **DFS đệ quy mang theo độ sâu** → **DFS lặp bằng ngăn xếp**.
4. Hiểu mấu chốt: **mọi nút cùng tầng có cùng độ sâu**; BFS đi tầng này xong mới sang tầng sau, còn DFS vẫn đúng nếu ta **ghi vào `ans[độ sâu]`**.
5. Tránh các lỗi kinh điển: quên `root is None`, `children` là `None` ở lá, **sửa hàng đợi trong lúc đang duyệt nó**, dùng ngăn xếp thay hàng đợi làm lẫn thứ tự, `ans[depth]` chưa tồn tại (`IndexError`), và đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho một **cây N-phân**, trả về **duyệt theo thứ tự từng tầng (level order)** các giá trị của nút: một list con cho mỗi tầng, từ tầng gốc xuống, trong mỗi tầng đọc **từ trái sang phải** (theo thứ tự các con trong `children`).

Đầu vào của LeetCode được **tuần tự hóa** theo level order: giá trị gốc, rồi `null`, rồi **từng nhóm con** của lần lượt các nút (theo thứ tự level order), mỗi nhóm kết thúc bằng `null`.

**Ví dụ 1:** `root = [1,null,3,2,4,null,5,6]` → `[[1],[3,2,4],[5,6]]`. Gốc `1` có ba con `3,2,4`; nút `3` có hai con `5,6`.

**Ví dụ 2:** `root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]` → `[[1],[2,3,4,5],[6,7,8,9,10],[11,12,13],[14]]`.

**Ràng buộc:** chiều cao cây `≤ 1000`; tổng số nút trong `[0, 10^4]`. (Cây rỗng: `root = None` → `[]`.)

Vì LeetCode đã dựng sẵn `Node`, trong notebook ta **tự dựng** `Node` và hàm `build_nary` (mục 3.2) để chạy ví dụ.

In [1]:
examples_lists = [
    ([1, None, 3, 2, 4, None, 5, 6], [[1], [3, 2, 4], [5, 6]]),
    ([1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14],
     [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]]),
    ([], []),
]
for lst, expected in examples_lists:
    print(lst, "->", expected)

[1, None, 3, 2, 4, None, 5, 6] -> [[1], [3, 2, 4], [5, 6]]
[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14] -> [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]]
[] -> []


## 3. Kiến thức nền

### 3.1. Nút N-phân `Node`

Cây nhị phân có `left/right`; cây N-phân có **một list `children`**. Nút lá có `children` **rỗng** (`[]`) — nhưng đề (và đôi khi LeetCode) có thể để `None`, nên code an toàn nên viết `node.children or []`.

In [2]:
class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children

    def __repr__(self):
        return f"Node({self.val})"


leaf5, leaf6 = Node(5, []), Node(6, [])
n3 = Node(3, [leaf5, leaf6])
root = Node(1, [n3, Node(2, []), Node(4, [])])
print(root.val, [c.val for c in root.children], [c.val for c in root.children[0].children])

print(Node(9).children)                 # None: mặc định (cẩn thận khi duyệt!)
print(Node(9).children or [])           # []  : cách phòng thủ

1 [3, 2, 4] [5, 6]
None
[]


### 3.2. Dựng cây từ list kiểu LeetCode: `build_nary`

Quy tắc đọc `[1,null,3,2,4,null,5,6]`:

1. Phần tử đầu là **gốc**; theo sau là một `null`.
2. Từ đó trở đi: lần lượt các **nhóm con**; mỗi nhóm là các giá trị **liên tiếp** kết thúc bằng `null`. Nhóm thứ 1 thuộc về **nút thứ 1** theo level order (là gốc), nhóm thứ 2 thuộc về nút thứ 2 (tức con đầu tiên của gốc), ... Nút không có con có nhóm **rỗng** (chỉ một `null`).

Ta dùng hàng đợi các nút "đang chờ nhận nhóm con": mỗi lần lấy nút ra, đọc nhóm con kế tiếp cho nó và xếp các con mới vào hàng đợi.

In [3]:
from collections import deque

def build_nary(lst):
    if not lst or lst[0] is None:
        return None
    root = Node(lst[0], [])
    q = deque([root])
    i = 2                                  # bỏ qua lst[1] (là null sau gốc)
    while q and i < len(lst):
        parent = q.popleft()
        while i < len(lst) and lst[i] is not None:
            child = Node(lst[i], [])
            parent.children.append(child)
            q.append(child)
            i += 1
        i += 1                              # bỏ qua null kết thúc nhóm
    return root


def nary_to_list(root):
    """Ngược lại: tuần tự hóa kiểu LeetCode (cắt null thừa ở cuối)."""
    if root is None:
        return []
    out = [root.val, None]
    q = deque([root])
    while q:
        node = q.popleft()
        for c in node.children or []:
            out.append(c.val)
            q.append(c)
        out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


t = build_nary([1, None, 3, 2, 4, None, 5, 6])
print([c.val for c in t.children], [c.val for c in t.children[0].children])
print(nary_to_list(t))
for lst, _ in examples_lists:
    assert nary_to_list(build_nary(lst)) == lst
print("khứ hồi build_nary / nary_to_list: OK")

[3, 2, 4] [5, 6]
[1, None, 3, 2, 4, None, 5, 6]
khứ hồi build_nary / nary_to_list: OK


### 3.3. Hàng đợi `deque`

**Hàng đợi (queue)**: vào sau ra sau (FIFO). Trong Python dùng `collections.deque`: `append(x)` thêm vào **cuối**, `popleft()` lấy từ **đầu** — cả hai `O(1)`. `extend(iterable)` thêm nhiều phần tử vào cuối. Đừng dùng `list.pop(0)`: nó **dịch toàn bộ** phần tử còn lại, `O(n)`.

In [4]:
from collections import deque
q = deque([1])
q.append(2); q.extend([3, 4])
print(q)                  # deque([1, 2, 3, 4])
print(q.popleft(), q)     # 1 deque([2, 3, 4])
print(len(q), bool(q))    # 3 True   (deque rỗng thì bool(q) == False)

deque([1, 2, 3, 4])
1 deque([2, 3, 4])
3 True


### 3.4. "Chụp" kích thước tầng: `for _ in range(len(q))`

Trong BFS, khi bắt đầu xử lý một tầng, hàng đợi chứa **đúng các nút của tầng đó**. Nếu ta lấy `k = len(q)` **trước**, rồi lấy ra đúng `k` nút, thì ta xử lý **trọn một tầng**, còn các con vừa xếp vào hàng đợi (tầng sau) **không bị lẫn**. Lưu ý: `range(len(q))` chỉ tính `len(q)` **một lần** lúc bắt đầu vòng `for`, nên việc `q` dài ra sau đó không ảnh hưởng.

In [5]:
q = deque(["a", "b"])
for _ in range(len(q)):          # len(q) = 2 được chốt ngay từ đầu
    x = q.popleft()
    q.append(x.upper())          # q thay đổi trong lúc lặp, nhưng vòng for vẫn chỉ chạy 2 lần
print(q)                         # deque(['A', 'B'])

deque(['A', 'B'])


### 3.5. List comprehension lồng nhau — "làm phẳng" danh sách các danh sách

`[c for node in level for c in node.children]` đọc là: "với mỗi `node` trong `level`, với mỗi `c` trong `node.children`, lấy `c`". Thứ tự `for` viết **từ ngoài vào trong**, giống hai vòng `for` lồng nhau. Mẫu này cho ta **tầng kế tiếp** chỉ bằng một dòng.

In [6]:
level = [n3, Node(2, []), Node(4, [])]
nxt = [c for node in level for c in node.children]
print(nxt)                           # [Node(5), Node(6)]
print([node.val for node in level])  # [3, 2, 4]

[Node(5), Node(6)]
[3, 2, 4]


### 3.6. Độ sâu (depth) của nút

Gốc có độ sâu `0`; con của nút độ sâu `d` có độ sâu `d + 1`. **Tầng thứ `d`** = mọi nút có độ sâu `d`. Một hàm đệ quy mang theo `depth` làm đúng việc này. Ví dụ in độ sâu mọi nút của cây ví dụ 1:

In [7]:
def print_depths(node, depth=0):
    print("  " * depth + f"{node.val} (độ sâu {depth})")
    for c in node.children or []:
        print_depths(c, depth + 1)

print_depths(build_nary([1, None, 3, 2, 4, None, 5, 6]))

1 (độ sâu 0)
  3 (độ sâu 1)
    5 (độ sâu 2)
    6 (độ sâu 2)
  2 (độ sâu 1)
  4 (độ sâu 1)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cây ví dụ 1: `1` có con `3, 2, 4`; `3` có con `5, 6`.

```
        1
     /  |  \
    3   2   4
   / \
  5   6
```

Tính tay theo **tầng**:

| Tầng | Các nút | Cách có được |
|---|---|---|
| 0 | `1` | gốc |
| 1 | `3, 2, 4` | các con của tầng 0, đọc theo thứ tự `children` |
| 2 | `5, 6` | các con của `3`, rồi của `2` (không có), rồi của `4` (không có) |

Quy tắc: **tầng `d+1` = nối các `children` của mọi nút tầng `d`, theo thứ tự trái → phải**. Dừng khi tầng rỗng. Mọi cách giải về cơ bản là các cách khác nhau để **thực hiện quy tắc này** (BFS) hoặc **gán nhãn độ sâu rồi gom** (DFS).

## 5. Cách 1 — Brute Force / baseline: thu (độ sâu, giá trị) cho mọi nút rồi gom nhóm

**Ý tưởng.** Tách làm hai bước dễ hiểu nhất:

1. **Duyệt cây** (DFS tiền thứ tự: nút trước, rồi các con từ trái sang phải) và ghi mỗi nút thành cặp `(độ sâu, giá trị)`.
2. **Sắp xếp ổn định theo độ sâu** (`sorted(..., key=lambda p: p[0])` giữ nguyên thứ tự gặp trong tiền thứ tự giữa các phần tử cùng khóa) rồi **gom** theo độ sâu.

Vì tiền thứ tự thăm các nút cùng tầng theo đúng thứ tự trái → phải, nên sau sắp xếp ổn định mỗi tầng vẫn giữ thứ tự đúng.

In [8]:
def level_order_pairs(root):
    if root is None:
        return []
    pairs = []

    def dfs(node, depth):
        pairs.append((depth, node.val))
        for c in node.children or []:
            dfs(c, depth + 1)

    dfs(root, 0)
    pairs.sort(key=lambda p: p[0])            # sort ổn định: giữ thứ tự trong cùng độ sâu
    ans = []
    for depth, val in pairs:
        if depth == len(ans):
            ans.append([])
        ans[depth].append(val)
    return ans


for lst, expected in examples_lists:
    print(level_order_pairs(build_nary(lst)), "(kỳ vọng", expected, ")")

[[1], [3, 2, 4], [5, 6]] (kỳ vọng [[1], [3, 2, 4], [5, 6]] )
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] (kỳ vọng [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] )
[] (kỳ vọng [] )


**Độ phức tạp (Cách 1).** Duyệt `O(n)` nhưng **sắp xếp** `O(n log n)` ⇒ Time **`O(n log n)`**. Space `O(n)` cho `pairs` (+ `O(h)` ngăn xếp đệ quy).

**Nút thắt:** sắp xếp là thừa. Ta đã biết độ sâu là số **nguyên nhỏ** `0..h`, có thể bỏ thẳng vào ô `ans[độ sâu]` mà không cần sắp xếp (Cách 4); hoặc đi **theo tầng** để khỏi cần độ sâu (BFS).

## 6. Ý tưởng thuật toán / bước đột phá

Có hai cách nhìn:

- **BFS (duyệt theo tầng):** giữ **một hàng đợi** các nút cần thăm. Ở mỗi vòng, tầng hiện tại **chính là toàn bộ** nội dung hàng đợi; ta lấy hết chúng ra, ghi giá trị, và xếp **con** của chúng vào hàng đợi — đó là tầng kế tiếp. Không cần biết độ sâu, cũng không cần sắp xếp.
- **DFS có nhãn độ sâu:** mang theo `depth` khi đệ quy và bỏ `val` vào `ans[depth]` (tạo ô mới khi `depth == len(ans)`). Vì ta thăm các con theo thứ tự trái → phải, thứ tự trong từng tầng vẫn đúng.

Cả hai đều **`O(n)`** thời gian: mỗi nút được thăm đúng một lần.

## 7. Các cách tiếp theo

### Cách 2 — BFS bằng hàng đợi, "chụp" kích thước tầng (lời giải chuẩn)

**Giải thích từng đoạn.**

1. Cây rỗng → trả `[]`.
2. `q = deque([root])`. Khi `q` còn nút: `t = []` là list giá trị của tầng; `for _ in range(len(q))`: `node = q.popleft()`, `t.append(node.val)`, `q.extend(node.children or [])`.
3. `ans.append(t)`.

In [9]:
def level_order_queue(root):
    ans = []
    if root is None:
        return ans
    q = deque([root])
    while q:
        t = []
        for _ in range(len(q)):               # đúng số nút của tầng hiện tại
            node = q.popleft()
            t.append(node.val)
            q.extend(node.children or [])     # con của node thuộc tầng kế tiếp
        ans.append(t)
    return ans


for lst, expected in examples_lists:
    print(level_order_queue(build_nary(lst)), "(kỳ vọng", expected, ")")

[[1], [3, 2, 4], [5, 6]] (kỳ vọng [[1], [3, 2, 4], [5, 6]] )
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] (kỳ vọng [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] )
[] (kỳ vọng [] )


In [10]:
class Solution2:
    def levelOrder(self, root: 'Node') -> list[list[int]]:
        ans = []
        if root is None:
            return ans
        q = deque([root])
        while q:
            t = []
            for _ in range(len(q)):
                node = q.popleft()
                t.append(node.val)
                q.extend(node.children or [])
            ans.append(t)
        return ans


print(Solution2().levelOrder(build_nary([1, None, 3, 2, 4, None, 5, 6])))

[[1], [3, 2, 4], [5, 6]]


**Độ phức tạp (Cách 2).** Mỗi nút vào và ra hàng đợi đúng một lần ⇒ Time **`O(n)`**. Space **`O(w)`** với `w` là độ rộng lớn nhất của một tầng (tệ nhất `O(n)`, ví dụ nút gốc có `n−1` con).

### Cách 3 — BFS theo "frontier" (không cần `deque`)

Thay vì hàng đợi chung, giữ **hai list**: `level` (tầng hiện tại) và tính ngay tầng sau bằng list comprehension lồng (mục 3.5). Mỗi vòng: ghi `[node.val for node in level]`, rồi `level = [c for node in level for c in node.children or []]`. Khi `level` rỗng thì dừng. Đây là BFS "từng tầng một" rất ngắn gọn; "chụp kích thước tầng" ở đây là **ngầm** (mỗi tầng là một list riêng).

In [11]:
def level_order_frontier(root):
    ans = []
    level = [root] if root else []
    while level:
        ans.append([node.val for node in level])
        level = [c for node in level for c in (node.children or [])]
    return ans


for lst, expected in examples_lists:
    print(level_order_frontier(build_nary(lst)), "(kỳ vọng", expected, ")")

[[1], [3, 2, 4], [5, 6]] (kỳ vọng [[1], [3, 2, 4], [5, 6]] )
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] (kỳ vọng [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] )
[] (kỳ vọng [] )


In [12]:
class Solution3:
    def levelOrder(self, root: 'Node') -> list[list[int]]:
        ans = []
        level = [root] if root else []
        while level:
            ans.append([node.val for node in level])
            level = [c for node in level for c in (node.children or [])]
        return ans


print(Solution3().levelOrder(build_nary([1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14])))

[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]]


**Độ phức tạp (Cách 3).** Time **`O(n)`**; Space **`O(w)`** (hai list tầng, mỗi cái tối đa độ rộng tầng). Ưu điểm: rất ngắn, không phải nhớ `popleft`. Nhược điểm: ít "tổng quát" hơn khi cần mang thêm thông tin cho mỗi nút (ví dụ cha, khoảng cách).

### Cách 4 — DFS đệ quy mang theo độ sâu

**Giải thích từng đoạn.**

1. `dfs(node, depth)`: nếu `depth == len(ans)` thì **tạo tầng mới** `ans.append([])` (lần đầu ta chạm tầng đó).
2. `ans[depth].append(node.val)`.
3. Với mỗi con theo thứ tự trái → phải: `dfs(child, depth + 1)`.

Vì mọi nút tầng `d` được thăm theo thứ tự trái → phải (DFS thăm cây con trái hoàn toàn trước cây con phải), nên thứ tự trong `ans[d]` đúng.

In [13]:
def level_order_dfs(root):
    ans = []

    def dfs(node, depth):
        if depth == len(ans):                 # lần đầu tới độ sâu này -> mở tầng mới
            ans.append([])
        ans[depth].append(node.val)
        for c in node.children or []:
            dfs(c, depth + 1)

    if root is not None:
        dfs(root, 0)
    return ans


for lst, expected in examples_lists:
    print(level_order_dfs(build_nary(lst)), "(kỳ vọng", expected, ")")

[[1], [3, 2, 4], [5, 6]] (kỳ vọng [[1], [3, 2, 4], [5, 6]] )
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] (kỳ vọng [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] )
[] (kỳ vọng [] )


In [14]:
class Solution4:
    def levelOrder(self, root: 'Node') -> list[list[int]]:
        ans = []

        def dfs(node, depth):
            if depth == len(ans):
                ans.append([])
            ans[depth].append(node.val)
            for c in node.children or []:
                dfs(c, depth + 1)

        if root is not None:
            dfs(root, 0)
        return ans


print(Solution4().levelOrder(build_nary([1, None, 3, 2, 4, None, 5, 6])))

[[1], [3, 2, 4], [5, 6]]


**Độ phức tạp (Cách 4).** Time **`O(n)`**; Space **`O(h)`** ngăn xếp đệ quy (+ `O(n)` kết quả). Với **cây cao tới 1000** (đề cho phép), đệ quy sâu `~1000` có thể chạm giới hạn mặc định của Python — mục 12.4 sẽ thử.

### Cách 5 — DFS **lặp** bằng ngăn xếp tường minh

Giống Cách 4 nhưng ta tự quản ngăn xếp các cặp `(node, depth)`. Vì ngăn xếp lấy **cuối** ra trước (LIFO), muốn thăm con **trái trước** thì **đẩy con theo thứ tự ngược** (`reversed(children)`).

1. `stack = [(root, 0)]`.
2. `node, depth = stack.pop()`; mở tầng mới nếu cần; `ans[depth].append(node.val)`.
3. `for c in reversed(node.children)`: `stack.append((c, depth + 1))`.

In [15]:
def level_order_stack(root):
    ans = []
    if root is None:
        return ans
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth == len(ans):
            ans.append([])
        ans[depth].append(node.val)
        for c in reversed(node.children or []):    # đẩy ngược để con trái được pop trước
            stack.append((c, depth + 1))
    return ans


for lst, expected in examples_lists:
    print(level_order_stack(build_nary(lst)), "(kỳ vọng", expected, ")")

[[1], [3, 2, 4], [5, 6]] (kỳ vọng [[1], [3, 2, 4], [5, 6]] )
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] (kỳ vọng [[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]] )
[] (kỳ vọng [] )


In [16]:
class Solution5:
    def levelOrder(self, root: 'Node') -> list[list[int]]:
        ans = []
        if root is None:
            return ans
        stack = [(root, 0)]
        while stack:
            node, depth = stack.pop()
            if depth == len(ans):
                ans.append([])
            ans[depth].append(node.val)
            for c in reversed(node.children or []):
                stack.append((c, depth + 1))
        return ans


print(Solution5().levelOrder(build_nary([1, None, 3, 2, 4, None, 5, 6])))

[[1], [3, 2, 4], [5, 6]]


**Độ phức tạp (Cách 5).** Time **`O(n)`**; Space **`O(n)`** tệ nhất (ngăn xếp có thể chứa nhiều con chờ). Không bị giới hạn đệ quy, nên an toàn cho cây rất sâu.

## 8. TRACE từng bước

Trace **Cách 2 (BFS)** trên cây ví dụ 1, in nội dung hàng đợi ở đầu mỗi tầng và từng lần `popleft`.

In [17]:
def trace_bfs(root):
    ans = []
    q = deque([root])
    level_no = 0
    while q:
        size = len(q)
        print(f"--- tầng {level_no}: hàng đợi = {[n.val for n in q]}, chụp size = {size}")
        t = []
        for k in range(size):
            node = q.popleft()
            t.append(node.val)
            kids = [c.val for c in node.children]
            q.extend(node.children)
            print(f"  lấy {node.val}: t = {t}, thêm con {kids} -> hàng đợi = {[n.val for n in q]}")
        ans.append(t)
        level_no += 1
    return ans


result = trace_bfs(build_nary([1, None, 3, 2, 4, None, 5, 6]))
print("kết quả:", result)
assert result == [[1], [3, 2, 4], [5, 6]]

--- tầng 0: hàng đợi = [1], chụp size = 1
  lấy 1: t = [1], thêm con [3, 2, 4] -> hàng đợi = [3, 2, 4]
--- tầng 1: hàng đợi = [3, 2, 4], chụp size = 3
  lấy 3: t = [3], thêm con [5, 6] -> hàng đợi = [2, 4, 5, 6]
  lấy 2: t = [3, 2], thêm con [] -> hàng đợi = [4, 5, 6]
  lấy 4: t = [3, 2, 4], thêm con [] -> hàng đợi = [5, 6]
--- tầng 2: hàng đợi = [5, 6], chụp size = 2
  lấy 5: t = [5], thêm con [] -> hàng đợi = [6]
  lấy 6: t = [5, 6], thêm con [] -> hàng đợi = []
kết quả: [[1], [3, 2, 4], [5, 6]]


Bảng trạng thái BFS (ví dụ 1):

| Tầng | Hàng đợi lúc bắt đầu | `size` | Lấy ra lần lượt | Con được thêm | `t` cuối tầng |
|---|---|---|---|---|---|
| 0 | `[1]` | 1 | `1` | `3, 2, 4` | `[1]` |
| 1 | `[3, 2, 4]` | 3 | `3`, `2`, `4` | `5, 6` (của `3`); `2`, `4` không có con | `[3, 2, 4]` |
| 2 | `[5, 6]` | 2 | `5`, `6` | (không) | `[5, 6]` |
| — | `[]` | — | — | — | dừng |

Chú ý lúc lấy `3` hàng đợi còn `[2, 4]` rồi thêm `5, 6` → `[2, 4, 5, 6]`: `5, 6` đứng **sau** `2, 4` nên không lẫn tầng, nhờ ta chỉ lấy đúng `size = 3` nút.

Trace **Cách 4 (DFS có độ sâu)** để thấy `ans` lớn dần theo thứ tự thăm tiền thứ tự:

In [18]:
def trace_dfs(root):
    ans = []

    def dfs(node, depth):
        pad = "  " * depth
        opened = ""
        if depth == len(ans):
            ans.append([])
            opened = " (mở tầng mới)"
        ans[depth].append(node.val)
        print(f"{pad}thăm {node.val} ở độ sâu {depth}{opened} -> ans = {ans}")
        for c in node.children:
            dfs(c, depth + 1)

    dfs(root, 0)
    return ans


result = trace_dfs(build_nary([1, None, 3, 2, 4, None, 5, 6]))
assert result == [[1], [3, 2, 4], [5, 6]]

thăm 1 ở độ sâu 0 (mở tầng mới) -> ans = [[1]]
  thăm 3 ở độ sâu 1 (mở tầng mới) -> ans = [[1], [3]]
    thăm 5 ở độ sâu 2 (mở tầng mới) -> ans = [[1], [3], [5]]
    thăm 6 ở độ sâu 2 -> ans = [[1], [3], [5, 6]]
  thăm 2 ở độ sâu 1 -> ans = [[1], [3, 2], [5, 6]]
  thăm 4 ở độ sâu 1 -> ans = [[1], [3, 2, 4], [5, 6]]


## 9. Code nộp LeetCode

Bản nộp (Cách 2: BFS bằng hàng đợi). Trên LeetCode cần `from collections import deque` (thường đã có sẵn); và **không** khai báo lại `Node`.

In [19]:
class Solution:
    def levelOrder(self, root: 'Node') -> list[list[int]]:
        ans = []
        if root is None:
            return ans
        q = deque([root])
        while q:
            t = []
            for _ in range(len(q)):
                node = q.popleft()
                t.append(node.val)
                q.extend(node.children or [])
            ans.append(t)
        return ans


for lst, expected in examples_lists:
    out = Solution().levelOrder(build_nary(lst))
    print(out)
    assert out == expected

[[1], [3, 2, 4], [5, 6]]
[[1], [2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13], [14]]
[]


## 10. Vì sao thuật toán đúng

**BFS (Cách 2).** Bất biến: **ngay trước mỗi vòng `while`, hàng đợi chứa đúng các nút của tầng `d`, theo thứ tự trái → phải.** Ban đầu `d = 0` và hàng đợi là `[root]`. Trong vòng `for`, ta lấy đúng `len(q)` nút (đúng cả tầng `d`) và xếp con của từng nút vào **cuối** hàng đợi, theo thứ tự nút cha rồi theo thứ tự con. Các con này chính là tầng `d+1`, theo thứ tự trái → phải (con của nút trái nằm trước con của nút phải). Vậy bất biến giữ cho `d+1`; vòng lặp kết thúc khi tầng rỗng ⇒ mọi nút được ghi đúng một lần ở đúng tầng.

**DFS (Cách 4, 5).** Mỗi nút ở độ sâu `d` được thêm vào `ans[d]`, nên đúng tầng. Về thứ tự trong tầng: giả sử `u` ở bên trái `v` (cùng tầng). Gọi `w` là tổ tiên chung thấp nhất; `u` thuộc nhánh con đứng trước nhánh chứa `v`, và DFS thăm hoàn toàn nhánh trước rồi mới sang nhánh sau, nên `u` được thêm vào trước `v`. Cách 5 đẩy con theo thứ tự ngược để `pop` ra con trái trước, đúng thứ tự thăm của Cách 4.

**Cách 1** chỉ là DFS rồi sắp xếp ổn định theo độ sâu: giữ thứ tự thăm trong cùng độ sâu ⇒ đúng. **Cách 3** thực hiện đúng quy tắc "tầng sau = nối `children` của tầng trước".

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. (độ sâu, giá trị) rồi sắp xếp | `O(n log n)` | `O(n)` |
| 2. BFS `deque` + chụp size | `O(n)` | `O(w)` |
| 3. BFS frontier | `O(n)` | `O(w)` |
| 4. DFS đệ quy + độ sâu | `O(n)` | `O(h)` |
| 5. DFS lặp + ngăn xếp | `O(n)` | `O(n)` tệ nhất |

`n` = số nút (`≤ 10^4`), `w` = độ rộng lớn nhất của một tầng, `h` = chiều cao (`≤ 1000`). Kết quả trả về luôn chiếm `O(n)` (không tính vào "space phụ" trong bảng).

## 12. Bộ test

Ta cần một **oracle độc lập** với các cách duyệt. Với test ngẫu nhiên, ta sinh cây từ **mảng cha** `parent[i]` (nút `i` là con của `parent[i] < i`, con được thêm theo thứ tự `i` tăng), và tính đáp án **không duyệt cây**: nút `i` có `depth[i] = depth[parent[i]] + 1` và **đường đi** `path[i]` = đường của cha cộng thêm chỉ số con. Thứ tự trái → phải trong một tầng chính là thứ tự từ điển của `path`; ta sắp theo `(depth, path)` rồi gom nhóm.

In [20]:
import random, sys

def make_random_tree(n, rng, max_children=None):
    """Sinh cây n nút: trả (root, expected_levels). Giá trị nút ngẫu nhiên 0..10^4 (có thể trùng)."""
    vals = [rng.randint(0, 10000) for _ in range(n)]
    nodes = [Node(vals[0], [])]
    depth, path = [0], [()]
    for i in range(1, n):
        while True:
            p = rng.randrange(i)
            if max_children is None or len(nodes[p].children) < max_children:
                break
        node = Node(vals[i], [])
        path.append(path[p] + (len(nodes[p].children),))
        depth.append(depth[p] + 1)
        nodes[p].children.append(node)
        nodes.append(node)
    order = sorted(range(n), key=lambda i: (depth[i], path[i]))
    expected = []
    for i in order:
        if depth[i] == len(expected):
            expected.append([])
        expected[depth[i]].append(vals[i])
    return nodes[0], expected


ALL = {
    "cach1": level_order_pairs,
    "cach2": level_order_queue,
    "cach3": level_order_frontier,
    "cach4": level_order_dfs,
    "cach5": level_order_stack,
    "nop":   lambda r: Solution().levelOrder(r),
}

def run_all(root, expected):
    for name, fn in ALL.items():
        assert fn(root) == expected, (name, fn(root), expected)

for lst, expected in examples_lists:
    run_all(build_nary(lst), expected)
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.1. Các trường hợp biên

In [21]:
# Cây rỗng
run_all(None, [])
# Một nút
run_all(Node(7, []), [[7]])
run_all(build_nary([7]), [[7]])
# Lá có children = None (như LeetCode đôi khi dựng)
r = Node(1, [Node(2), Node(3)])           # 2 và 3 có children=None
run_all(r, [[1], [2, 3]])
# Chuỗi (mỗi nút một con)
chain = Node(0, [])
cur = chain
for v in range(1, 6):
    nxt = Node(v, []); cur.children.append(nxt); cur = nxt
run_all(chain, [[0], [1], [2], [3], [4], [5]])
# Giá trị trùng nhau và giá trị 0
run_all(build_nary([0, None, 0, 0, None, 0]), [[0], [0, 0], [0]])
# Gốc có rất nhiều con
wide = Node(1, [Node(i, []) for i in range(2, 10001)])
run_all(wide, [[1], list(range(2, 10001))])
print("OK")

OK


### 12.2. Phản ví dụ cho các cách làm sai

Ba hàm sai điển hình (giải thích ở mục 13); test cho thấy chúng không qua được ví dụ 1:

In [22]:
def wrong_stack_as_queue(root):          # dùng list.pop() (lấy cuối) thay vì popleft
    ans = []
    if root is None:
        return ans
    q = [root]
    while q:
        t = []
        for _ in range(len(q)):
            node = q.pop()               # SAI: LIFO làm đảo thứ tự trong tầng
            t.append(node.val)
            q.extend(node.children)
        ans.append(t)
    return ans

def wrong_no_reverse(root):              # DFS lặp nhưng quên đảo thứ tự con
    ans = []
    if root is None:
        return ans
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth == len(ans):
            ans.append([])
        ans[depth].append(node.val)
        for c in node.children:          # SAI: con phải sẽ được pop trước
            stack.append((c, depth + 1))
    return ans

def wrong_flat_bfs(root):                # BFS nhưng gộp mọi nút vào một list, không tách tầng
    out = []
    q = deque([root])
    while q:
        node = q.popleft()
        out.append(node.val)
        q.extend(node.children)
    return [out]

ex1 = [1, None, 3, 2, 4, None, 5, 6]
for fn in (wrong_stack_as_queue, wrong_no_reverse, wrong_flat_bfs):
    got = fn(build_nary(ex1))
    print(fn.__name__, "->", got)
    assert got != [[1], [3, 2, 4], [5, 6]]

wrong_stack_as_queue -> [[1], [4, 2, 3], [6, 5]]
wrong_no_reverse -> [[1], [4, 2, 3], [6, 5]]
wrong_flat_bfs -> [[1, 3, 2, 4, 5, 6]]


### 12.3. Đối chiếu năm cách trên cây ngẫu nhiên (seed cố định)

Sinh cây ngẫu nhiên đủ kiểu (nhiều con, ít con, giới hạn số con) và so với **oracle** từ mảng cha. Đồng thời kiểm tra khứ hồi `build_nary(nary_to_list(root))`.

In [23]:
rng = random.Random(429)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 80)
    root, expected = make_random_tree(n, rng, max_children=rng.choice([None, 1, 2, 3, 5]))
    run_all(root, expected)
    again = build_nary(nary_to_list(root))
    assert level_order_queue(again) == expected
    cnt += 1
print("đã đối chiếu", cnt, "cây ngẫu nhiên: OK")

# cây lớn: 10^4 nút
root, expected = make_random_tree(10000, rng, max_children=4)
run_all(root, expected)
print("cây 10^4 nút: OK, số tầng =", len(expected))

đã đối chiếu 400 cây ngẫu nhiên: OK


cây 10^4 nút: OK, số tầng = 23


### 12.4. Cây rất sâu (chiều cao 1000): đệ quy hay lặp?

Đề cho phép chiều cao đến `1000`. Đệ quy sâu 1000 tầng dễ chạm giới hạn đệ quy mặc định (~1000) của Python. Các cách lặp (2, 3, 5) không gặp vấn đề.

In [24]:
chain = Node(0, [])
cur = chain
for v in range(1, 1000):
    nxt = Node(v, []); cur.children.append(nxt); cur = nxt
expected = [[v] for v in range(1000)]

for name in ("cach2", "cach3", "cach5", "nop"):
    assert ALL[name](chain) == expected, name
print("các cách lặp trên chuỗi sâu 1000: OK")

try:
    level_order_dfs(chain)
    print("DFS đệ quy qua được ở giới hạn mặc định")
except RecursionError:
    print("DFS đệ quy: RecursionError ở giới hạn mặc định (như dự đoán)")

old = sys.getrecursionlimit()
sys.setrecursionlimit(5000)
assert level_order_dfs(chain) == expected
sys.setrecursionlimit(old)
print("sau khi nâng recursion limit: OK")

các cách lặp trên chuỗi sâu 1000: OK
DFS đệ quy: RecursionError ở giới hạn mặc định (như dự đoán)
sau khi nâng recursion limit: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sửa hàng đợi trong lúc đang `for` qua chính nó

In [25]:
def wrong_iterate_queue(root):
    ans, q = [], deque([root])
    while q:
        t = []
        for node in q:                    # SAI: duyệt trực tiếp q ...
            t.append(node.val)
            q.extend(node.children)       # ... trong khi lại sửa q
        ans.append(t)
        q.clear()
    return ans

try:
    wrong_iterate_queue(build_nary([1, None, 3, 2, 4, None, 5, 6]))
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: deque mutated during iteration


**Vì sao sai:** thêm/xóa phần tử của `deque` trong khi đang duyệt nó là **không được phép** (và với `list` thì kết quả khó đoán). **Đúng:** chụp `len(q)` rồi `popleft()` đúng từng ấy lần (Cách 2), hoặc làm hai list `level`/`next_level` riêng biệt (Cách 3).

### Lỗi 2: Nút lá có `children = None`

In [26]:
def wrong_none_children(root):
    ans, q = [], deque([root])
    while q:
        t = []
        for _ in range(len(q)):
            node = q.popleft()
            t.append(node.val)
            q.extend(node.children)          # SAI nếu children là None
        ans.append(t)
    return ans

try:
    wrong_none_children(Node(1, [Node(2), Node(3)]))    # 2, 3 có children=None
except TypeError as e:
    print("TypeError:", e)
print("đúng:", level_order_queue(Node(1, [Node(2), Node(3)])))

TypeError: 'NoneType' object is not iterable
đúng: [[1], [2, 3]]


**Vì sao sai:** `deque.extend(None)` lỗi (`None` không lặp được). **Đúng:** `q.extend(node.children or [])` — an toàn dù `children` là `[]` hay `None`.

### Lỗi 3: Quên cây rỗng

In [27]:
def wrong_no_empty_check(root):
    q = deque([root])
    node = q.popleft()
    return node.val                          # root là None -> AttributeError

try:
    wrong_no_empty_check(None)
except AttributeError as e:
    print("AttributeError:", e)
print("đúng:", Solution().levelOrder(None))

AttributeError: 'NoneType' object has no attribute 'val'
đúng: []


**Vì sao sai:** đề cho phép `0` nút (`root = None`). **Đúng:** `if root is None: return []` ở đầu (hoặc `level = [root] if root else []` trong Cách 3).

### Lỗi 4: Dùng ngăn xếp (LIFO) thay cho hàng đợi (FIFO) trong BFS

In [28]:
print("sai :", wrong_stack_as_queue(build_nary([1, None, 3, 2, 4, None, 5, 6])))
print("đúng:", level_order_queue(build_nary([1, None, 3, 2, 4, None, 5, 6])))

sai : [[1], [4, 2, 3], [6, 5]]
đúng: [[1], [3, 2, 4], [5, 6]]


**Vì sao sai:** `list.pop()` lấy phần tử **cuối** nên tầng `[3, 2, 4]` bị đọc ngược thành `[4, 2, 3]`. BFS cần **FIFO**: `deque.popleft()` (hoặc `list.pop(0)` nhưng chậm `O(n)`).

### Lỗi 5: `ans[depth]` chưa tồn tại (`IndexError`) trong DFS

In [29]:
def wrong_dfs_no_open(root):
    ans = []
    def dfs(node, depth):
        ans[depth].append(node.val)          # SAI: tầng chưa được tạo
        for c in node.children:
            dfs(c, depth + 1)
    dfs(root, 0)
    return ans

try:
    wrong_dfs_no_open(build_nary([1, None, 2]))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** lần đầu chạm độ sâu `d`, `ans` mới có `d` phần tử (chỉ số `0..d-1`). **Đúng:** `if depth == len(ans): ans.append([])` trước khi `ans[depth].append(...)` (do DFS đi sâu từng bước một nên `depth` không bao giờ vượt `len(ans)` quá 0).

## 14. Mẹo & nhận dạng pattern

- **"Theo từng tầng / level by level / khoảng cách tính theo số cạnh" ⇒ BFS.** Công thức chuẩn: `while q: for _ in range(len(q)): ...`.
- **"Số bước ít nhất" trên đồ thị không trọng số ⇒ BFS** (xem bài 433 Minimum Genetic Mutation ở cùng nhóm).
- Muốn mang thêm thông tin cho mỗi nút (độ sâu, cha) ⇒ xếp **tuple** `(node, depth)` vào hàng đợi.
- DFS cũng ra level order nếu bạn **gán nhãn độ sâu** — hữu ích khi cần đồng thời thông tin "cây con".
- **Hàng đợi = `deque`**, không `list.pop(0)`.
- `children or []` — thói quen phòng thủ khi nút lá có `children` là `None`.
- Cây có thể **rất sâu** ⇒ cân nhắc bản lặp để khỏi `RecursionError`.

## 15. Biến thể

1. **Binary Tree Level Order Traversal (102)** — phiên bản cây nhị phân (hai con `left/right`).
2. **Zigzag Level Order (103)** — tầng chẵn đọc trái → phải, tầng lẻ phải → trái.
3. **Maximum Depth of N-ary Tree (559)** — số tầng của cây N-phân (`len(level_order)`).
4. **N-ary Tree Preorder / Postorder Traversal (589 / 590).**
5. **Average of Levels (637)**, **Largest Value in Each Row (515)** — tính gì đó trên mỗi tầng.
6. **Serialize and Deserialize N-ary Tree (428)** — chính là `build_nary` / `nary_to_list` ở mục 3.2.
7. **Bottom-up level order** — đảo ngược `ans` ở cuối.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cặp (độ sâu, giá trị) + sắp xếp | DFS ghi nhãn rồi `sort` ổn định | `O(n log n)` | `O(n)` | hiểu đề, oracle đơn giản |
| 2. BFS `deque` + chụp size | mỗi vòng lấy trọn một tầng | `O(n)` | `O(w)` | **nên nộp** |
| 3. BFS frontier | tầng sau = nối `children` của tầng trước | `O(n)` | `O(w)` | ngắn gọn, không cần `deque` |
| 4. DFS đệ quy + độ sâu | `ans[depth].append(val)` | `O(n)` | `O(h)` | khi cần tư duy cây con |
| 5. DFS lặp + ngăn xếp | như 4, không đệ quy | `O(n)` | `O(n)` | cây rất sâu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().levelOrder(build_nary([1, None, 2, 3]))`; (b) `Solution().levelOrder(build_nary([5]))`; (c) `Solution().levelOrder(None)`; (d) `nary_to_list(build_nary([1, None, 2, 3, None, 4]))`.

**Bài 2 (tính tay).** Với `root = [1,null,2,3,4,null,5,null,6,7]`: vẽ cây (chú ý: nhóm con thứ 2 là của nút `2`, nhóm thứ 3 của nút `3`, ...), rồi viết kết quả level order và nội dung hàng đợi đầu mỗi tầng.

**Bài 3 (điền TODO).** Hoàn thành `level_order_todo(root)` theo Cách 2.

In [30]:
def level_order_todo(root):
    # TODO 1: nếu root là None thì trả về []
    ans = []
    # TODO 2: q = deque([root])
    # TODO 3: while q: tạo t = []; for _ in range(len(q)): popleft, t.append(node.val), q.extend(node.children or [])
    # TODO 4: sau mỗi tầng ans.append(t)
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `max_depth_nary(root)` (bài 559) bằng **hai cách**: (a) `len(Solution().levelOrder(root))`; (b) đệ quy `1 + max(max_depth(c) for c in children)` (cẩn thận lá và cây rỗng). Kiểm tra chúng bằng nhau trên vài cây.

**Bài 5 (biến thể / mini challenge).** (a) **Zigzag** cho cây N-phân: `zigzag_nary(root)` — tầng chỉ số lẻ đảo ngược. (b) **Largest Value in Each Row** cho cây N-phân: `row_max(root)` trả list max từng tầng. Kiểm tra với ví dụ 1: zigzag → `[[1],[4,2,3],[5,6]]`, row_max → `[1,4,6]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[[1], [2, 3]]`; `[[5]]`; `[]`; `[1, None, 2, 3, None, 4]` (nhóm con thứ hai là của nút `2`, con `4`). Bài 3: nhớ `return []` cho cây rỗng.
- **Mức 2 (vừa):** Bài 2: gốc `1` có `2,3,4`; nút `2` có con `5`; nút `3` có con `6, 7`; nút `4` không có con ⇒ `[[1],[2,3,4],[5,6,7]]`. Bài 4(b): quy ước cây rỗng `0`, lá `1`.
- **Mức 3 (sâu):** Bài 5: dùng kết quả `levelOrder` rồi `for i, row in enumerate(levels): if i % 2: row.reverse()`; row_max: `[max(row) for row in levels]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert Solution().levelOrder(build_nary([1, None, 2, 3])) == [[1], [2, 3]]
assert Solution().levelOrder(build_nary([5])) == [[5]]
assert Solution().levelOrder(None) == []
assert nary_to_list(build_nary([1, None, 2, 3, None, 4])) == [1, None, 2, 3, None, 4]

# Bài 2
r2 = build_nary([1, None, 2, 3, 4, None, 5, None, 6, 7])
assert Solution().levelOrder(r2) == [[1], [2, 3, 4], [5, 6, 7]]


# Bài 3
def level_order_todo_answer(root):
    if root is None:
        return []
    ans = []
    q = deque([root])
    while q:
        t = []
        for _ in range(len(q)):
            node = q.popleft()
            t.append(node.val)
            q.extend(node.children or [])
        ans.append(t)
    return ans

for lst, expected in examples_lists:
    assert level_order_todo_answer(build_nary(lst)) == expected


# Bài 4
def max_depth_levels(root):
    return len(Solution().levelOrder(root))

def max_depth_rec(root):
    if root is None:
        return 0
    return 1 + max((max_depth_rec(c) for c in root.children or []), default=0)

for lst, expected in examples_lists:
    r = build_nary(lst)
    assert max_depth_levels(r) == max_depth_rec(r) == len(expected)


# Bài 5
def zigzag_nary(root):
    levels = Solution().levelOrder(root)
    for i, row in enumerate(levels):
        if i % 2:
            row.reverse()
    return levels

def row_max(root):
    return [max(row) for row in Solution().levelOrder(root)]

ex1_root = build_nary([1, None, 3, 2, 4, None, 5, 6])
assert zigzag_nary(ex1_root) == [[1], [4, 2, 3], [5, 6]]
assert row_max(ex1_root) == [1, 4, 6]
assert zigzag_nary(None) == [] and row_max(None) == []
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (BFS theo tầng bằng `deque`)

```python
class Solution:
    def levelOrder(self, root: 'Node') -> List[List[int]]:
        ans = []
        if root is None:
            return ans
        q = deque([root])
        while q:
            t = []
            for _ in range(len(q)):
                node = q.popleft()
                t.append(node.val)
                q.extend(node.children or [])
            ans.append(t)
        return ans
```

**Checklist**

- [ ] Có `if root is None: return []` ở đầu?
- [ ] Dùng `deque` và `popleft()` (FIFO), không dùng `pop()` (LIFO) hay `pop(0)` (chậm)?
- [ ] "Chụp" `len(q)` **trước** khi lấy phần tử, để tách đúng từng tầng?
- [ ] Dùng `node.children or []` (an toàn khi lá có `children = None`)?
- [ ] Không sửa hàng đợi trong lúc `for node in q`?
- [ ] Nếu làm DFS: có `if depth == len(ans): ans.append([])` trước `ans[depth].append(...)`, và (DFS lặp) đẩy con theo thứ tự **ngược**?
- [ ] Không khai báo lại `Node` trong bài nộp; đã thử cây rỗng, một nút, chuỗi sâu, gốc nhiều con?